In [ ]:
import cosmographi as cp
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from contextlib import closing
import pandas as pd
from time import time
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # pick whichever GPU has more free memory
np.random.seed(0)
key = jax.random.PRNGKey(0) #pseudorandom number generato rkey
colours = ["purple", "green", "red", "teal", "brown", "yellow"]

The first step is to extract the data from the LSST database using SQL.
Note that form the output, we can tell that each rown observation and each collumn is a variable. 

In [ ]:
# Extract survey data from the LSST database
# #####################################################################
survey_path = "/home/renee/renee/opsim/baseline_v5.0.1_10yrs.db"
with closing(sqlite3.connect(survey_path)) as conn:
    query = f"""
        SELECT 
            observationId,
            
            observationStartMJD,
            visitExposureTime,
            numExposures,
            airmass,
            fieldRA, 
            fieldDec, 
            rotTelPos, 
            rotSkyPos, 
            skyBrightness,
            band,
            filter,
            seeingFwhmEff,
            observation_reason
        FROM observations
        LIMIT 2000;
        """
    survey_df = pd.read_sql_query(query, conn) # all data will be togetehr in a pandas df
print(survey_df)
assert (
    survey_df["numExposures"].nunique() == 1
), "All exposures should have the same number of exposures."
survey_t = survey_df["observationStartMJD"].values
print(type(survey_t))
survey_ra = survey_df["fieldRA"].values
survey_dec = survey_df["fieldDec"].values
survey_exp_time = survey_df["visitExposureTime"].values
survey_airmass = survey_df["airmass"].values
survey_sky_brightness = survey_df["skyBrightness"].values
survey_seeing = survey_df["seeingFwhmEff"].values
survey_PSF_Aeff = 4 * np.pi * (survey_seeing / 2.355) ** 2  # convert FWHM to effective area
survey_band = survey_df["band"].values
survey_obsreason = survey_df["observation_reason"].values

      observationId  observationStartMJD  visitExposureTime  numExposures  \
0                 0         60981.002252               15.0             1   
1                 1         60981.002498               15.0             1   
2                 2         60981.002744               15.0             1   
3                 3         60981.002988               15.0             1   
4                 4         60981.003234               15.0             1   
...             ...                  ...                ...           ...   
1995           1995         60991.338102               30.0             1   
1996           1996         60991.340081               30.0             1   
1997           1997         60991.340527               30.0             1   
1998           1998         60991.340945               30.0             1   
1999           1999         60991.341370               30.0             1   

       airmass     fieldRA   fieldDec  rotTelPos   rotSkyPos  skyBrightness

## Setting up a 'source' to observe
We now need to have a source to observe. In this example we have coded up a Type Ia supernova that is specified by the [SALT2 2021](https://academic.oup.com/mnras/article/504/3/4111/6225808) light curve model, and have assumed Milky Way extinction according to the [Calzetti law](https://iopscience.iop.org/article/10.1086/308692).  We are assuming a standard LCDM cosmology.


In [ ]:
# Create source
C = cp.Cosmology()
# creates new class 
source_obj = cp.source_factory(cp.SALT2_2021, cp.source.effects.MWExtinction_Calzetti00) 
S = source_obj(cosmology=C, A_V_c00mw=0.1, R_V_c00mw=3.1) # dust extinction parameters
S.load_salt2_model()
S.M.to_static() 
S.CL.to_static()
print(S)

SALT2_2021_MWExtinction_Calzetti00|SALT2_2021_MWExtinction_Calzetti00
    z|dynamic
    mu|pointer
        Cosmology|Cosmology
            H0|static: 67.9
            Omega_m|static: 0.307
            Omega_k|static: 0
            Omega_r|static: 0
            Omega_l|pointer: 0.693
                Omega_m|static: 0.307
                Omega_k|static: 0
                Omega_r|static: 0
            w0|static: -1
            wa|static: 0
        z|dynamic
    t0|dynamic
    x1|dynamic
    c|dynamic
    M|static: (2, 71, 1441)
    CL|static: [-0.461, 0.754, -0.455, 0.0818]
    A_V_c00mw|static: 0.1
    R_V_c00mw|static: 3.1


## Setting up the instrument model

In [ ]:
I = cp.RubinObservatory()
I.throughput.trim()
I.throughput.to_dynamic() 
survey_iband = cp.utils.bandstr_to_bandidx(I.throughput.bands, survey_band)
nfilter = len(I.throughput.bands) # number of filters
print(I)
print(I.get_values().shape)

RubinObservatory|RubinObservatory
    Throughput_wAtmos|Throughput_wAtmos
        air_mass|dynamic: 1.2
    MagAB|MagAB
        Ze|static: (6,)
        Throughput_wAtmos|Throughput_wAtmos
            air_mass|dynamic: 1.2
(1,)


## Generate the SN source sample

Now we can pull our number of supernovae that we want to observe. Here we are 'forcing' the situation where we pick values of right ascension and declination that coincide with the observed point (so we don't waste computational resources). We also restrict our sample to those sources who can be 'observed' 5 times in order to get a good example of the lightcurve.

The default is Nsn = 300 but you can reduce this for testing purposes.

The ```sel_src``` is the list of the selected sources that meet our criteria.

In [ ]:
# Generate mock SN
Nsn = 30 # number of sources that we will simulate
t_range = (np.min(survey_t), np.max(survey_t)) # ranges of times we observe at
source_t = np.random.uniform(*t_range, Nsn) # draw 30 random source times (uniformly dist.)
S.t0 = source_t
R = cp.RateConst(cosmology=C, z_min=0.01, z_max=1.0, r=1) 
key, subkey = jax.random.split(key)
source_z = R.sample_z(subkey, source_t.shape) # sample redshifts
S.z = source_z
chooseraddec = np.random.choice(
    len(survey_ra), Nsn
)  # fixme, not right distribution, but good enough for testing
# sample from [0, len(survey_ra)] 30 times
print(chooseraddec)
source_ra, source_dec = cp.utils.sample_near(
    survey_ra[chooseraddec], survey_dec[chooseraddec], radius_deg=2, n=Nsn
) # choose random Ra and Dec and then sample near those points, within 2 degrees, to get source positions
print(len(source_ra))
print("Matching sources to survey...")
source_matches = cp.utils.cross_match_survey_circle(
    source_tmin=np.array(jax.vmap(lambda z, t: S.min_time({"z": z, "t0": t}))(source_z, source_t)),
    source_tmax=np.array(jax.vmap(lambda z, t: S.max_time({"z": z, "t0": t}))(source_z, source_t)),
    source_ra=source_ra,
    source_dec=source_dec,
    survey_t=survey_t,
    survey_ra=survey_ra,
    survey_dec=survey_dec,
    survey_fov=3.5,  # degrees, LSST field of view
) # see what is in our field of view when we are observing, match with sources if any
print(f"Matched {len(source_matches)} sources to survey observations.")
for i in list(source_matches.keys()):
    if len(source_matches[i]) < 5:
        del source_matches[
            i
        ]  # drop sources with fewer than 5 matches, not enough data for lightcurve
        continue
print(f"Matched {len(source_matches)} sources worth examining.")
P = cp.source.prior.SALT2_SK16Prior()
x1, c = P.prior_sample(source_t.shape[0])
S.x1 = x1
S.c = c
sel_src = np.sort(list(source_matches.keys()))

print(sel_src)

[1142 1195  474 1039 1860 1226   85 1703  106 1515 1630 1494  256 1617
 1210 1036 1243  811  950  381  431  790  799  372 1058 1834 1348 1914
  830 1828]
30
Matching sources to survey...
Matched 30 sources to survey observations.
Matched 22 sources worth examining.
[ 1  3  5  6  7  8  9 10 11 13 14 15 16 17 19 21 22 23 24 26 27 29]


In [ ]:
# Build params input
#####################################################################
source_values = np.array(S.get_values())
survey_values = np.array(I.get_values())
keys = []
bands = []
exp_times = []
sky_brightnesses = []
PSF_Aeffs = []
obs_t = []
obs_reasons = []
source_params = []
survey_params = []
for i in sel_src:
    _subkeys = jax.random.split(key, len(source_matches[i]) + 1)
    key, subkeys = _subkeys[0], _subkeys[1:]
    keys.append(subkeys)
    bands.append(survey_iband[source_matches[i]])
    exp_times.append(survey_exp_time[source_matches[i]])
    sky_brightnesses.append(survey_sky_brightness[source_matches[i]])
    PSF_Aeffs.append(survey_PSF_Aeff[source_matches[i]])
    obs_t.append(survey_t[source_matches[i]])
    obs_reasons.append(survey_obsreason[source_matches[i]])
    source_params.append(jnp.repeat(source_values[i][None, :], len(source_matches[i]), axis=0))
    survey_params.append(survey_airmass[source_matches[i]][:, None])  # shape (n_obs, 1)
keys = jnp.array(np.concatenate(keys))
bands = jnp.array(np.concatenate(bands))
exp_times = jnp.array(np.concatenate(exp_times))
sky_brightnesses = jnp.array(np.concatenate(sky_brightnesses))
PSF_Aeffs = jnp.array(np.concatenate(PSF_Aeffs))
obs_t = jnp.array(np.concatenate(obs_t))
obs_reasons = np.array(np.concatenate(obs_reasons))
source_params = jnp.array(np.concatenate(source_params, axis=0))
survey_params = jnp.array(np.concatenate(survey_params, axis=0))
print("Keys shape:", keys.shape)
print("Bands shape:", bands.shape)
print("Exp times shape:", exp_times.shape)
print("Source params shape:", source_params.shape)
print("Survey params shape:", survey_params.shape)